# Binder Contact Filter

Filters a set of AF2 multimer PDB files by whether the binder chain contacts a defined target epitope region.

## Pipeline position
```
RFDiffusion / BindCraft / Forge → PDB files (binder + target complex)
    ↓
[THIS NOTEBOOK] Contact filter
    • Keep: binder contacts target epitope residues
    • Discard: binder misses the epitope entirely
    → passing_binders.csv  +  summary table
    ↓
[Notebook 03] Full binder comparison + UMAP + motif analysis
```

## Inputs
- Zip file of PDB files (AF2 multimer format — binder + target as separate chains)
- Target chain ID, binder chain ID
- Epitope residue range(s) on the target you want the binder to contact
- Distance cutoff for defining a contact

## 0. Install dependencies

In [ ]:
%%capture
!pip install biopython pandas matplotlib

## 1. Configuration

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────

# Upload your zip of PDB files using the Files panel, then set the path here
PDB_ZIP       = "binders.zip"       # or set PDB_DIR if already unzipped
PDB_DIR       = "binder_pdbs"       # directory PDBs will be extracted to

TARGET_CHAIN  = "A"                 # chain ID of the target protein
BINDER_CHAIN  = "B"                 # chain ID of the binder

# Epitope ranges on the TARGET chain (residue numbers as in the PDB)
# Add as many ranges as you need
EPITOPE_RANGES = [
    (33, 38, "PHF6 VQIVYK"),
    (2,   7, "PHF6* VQIINK"),
]

# A binder PASSES if it has at least this many contacts to ANY epitope range
MIN_CONTACTS  = 1

# Cα–Cα distance cutoff for a contact (Å)
CONTACT_CUTOFF = 8.0

# Minimum mean pLDDT at the interface to consider the prediction reliable
# (AF2 stores pLDDT in the B-factor column)
MIN_INTERFACE_PLDDT = 70.0

# ────────────────────────────────────────────────────────────────────────────

epitope_positions = {}
for start, end, label in EPITOPE_RANGES:
    for r in range(start, end + 1):
        epitope_positions[r] = label

print(f"Target chain  : {TARGET_CHAIN}")
print(f"Binder chain  : {BINDER_CHAIN}")
print(f"Epitope sites : {EPITOPE_RANGES}")
print(f"Contact cutoff: {CONTACT_CUTOFF} Å")
print(f"Min contacts  : {MIN_CONTACTS}")
print(f"Min interface pLDDT: {MIN_INTERFACE_PLDDT}")

## 2. Extract PDB files

In [ ]:
import zipfile, os, glob

if os.path.exists(PDB_ZIP):
    os.makedirs(PDB_DIR, exist_ok=True)
    with zipfile.ZipFile(PDB_ZIP) as z:
        z.extractall(PDB_DIR)
    print(f"Extracted to {PDB_DIR}/")

pdb_files = sorted(glob.glob(os.path.join(PDB_DIR, "**/*.pdb"), recursive=True) +
                   glob.glob(os.path.join(PDB_DIR, "*.pdb")))

print(f"Found {len(pdb_files)} PDB files")

## 3. Parse PDBs and compute contacts

In [ ]:
import numpy as np
from Bio import PDB
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

parser = PDB.PDBParser(QUIET=True)

def get_ca(chain):
    """Return {res_id: (ca_coord, bfactor)} for all residues in chain."""
    atoms = {}
    for res in chain.get_residues():
        if res.get_id()[0] != ' ':  # skip HETATM
            continue
        res_num = res.get_id()[1]
        if 'CA' in res:
            ca = res['CA']
            atoms[res_num] = (ca.get_vector().get_array(), ca.get_bfactor())
    return atoms


def analyze_pdb(path):
    struct = parser.get_structure('s', path)
    model  = struct[0]

    if TARGET_CHAIN not in [c.id for c in model]:
        return None, f"Target chain {TARGET_CHAIN} not found"
    if BINDER_CHAIN not in [c.id for c in model]:
        return None, f"Binder chain {BINDER_CHAIN} not found"

    target_ca = get_ca(model[TARGET_CHAIN])
    binder_ca = get_ca(model[BINDER_CHAIN])

    if not target_ca or not binder_ca:
        return None, "Empty chain"

    # Find all contacts between binder and target
    contacts = []   # (binder_res, target_res, distance, epitope_label)
    epitope_contacts = defaultdict(list)

    for b_res, (b_coord, _) in binder_ca.items():
        for t_res, (t_coord, _) in target_ca.items():
            dist = np.linalg.norm(b_coord - t_coord)
            if dist <= CONTACT_CUTOFF:
                label = epitope_positions.get(t_res, None)
                contacts.append((b_res, t_res, round(dist, 2), label))
                if label:
                    epitope_contacts[label].append((b_res, t_res, round(dist, 2)))

    # Interface pLDDT: mean B-factor of binder residues that make any contact
    contacting_binder_res = set(c[0] for c in contacts)
    if contacting_binder_res:
        interface_plddt = np.mean([binder_ca[r][1] for r in contacting_binder_res
                                   if r in binder_ca])
    else:
        interface_plddt = 0.0

    # Per-epitope contact counts
    epitope_counts = {label: len(set((c[0], c[1]) for c in hits))
                      for label, hits in epitope_contacts.items()}
    total_epitope_contacts = sum(epitope_counts.values())

    result = {
        "total_contacts": len(contacts),
        "total_epitope_contacts": total_epitope_contacts,
        "epitope_counts": epitope_counts,
        "interface_plddt": round(interface_plddt, 1),
        "target_residues_contacted": sorted(set(c[1] for c in contacts)),
        "binder_length": len(binder_ca),
        "target_length": len(target_ca),
    }
    return result, None


# Run on all PDBs
print(f"Analyzing {len(pdb_files)} PDB files...")
records = []
for path in pdb_files:
    name = os.path.basename(path)
    result, err = analyze_pdb(path)
    if err:
        records.append({"pdb": name, "status": "ERROR", "error": err})
    else:
        records.append({"pdb": name, "path": path, **result, "error": ""})

print("Done.")

## 4. Apply filter and build results table

In [ ]:
import pandas as pd

rows = []
for r in records:
    if r.get("status") == "ERROR":
        rows.append({"pdb": r["pdb"], "verdict": "ERROR",
                     "reason": r["error"], "epitope_contacts": 0,
                     "total_contacts": 0, "interface_plddt": 0,
                     "binder_length": 0, "epitope_breakdown": ""})
        continue

    epi_contacts = r["total_epitope_contacts"]
    iface_plddt  = r["interface_plddt"]

    passes_contacts = epi_contacts >= MIN_CONTACTS
    passes_plddt    = iface_plddt  >= MIN_INTERFACE_PLDDT

    if passes_contacts and passes_plddt:
        verdict = "PASS"
        reason  = ""
    elif not passes_contacts:
        verdict = "DISCARD"
        reason  = f"only {epi_contacts} epitope contact(s) (need {MIN_CONTACTS})"
    else:
        verdict = "DISCARD"
        reason  = f"interface pLDDT={iface_plddt:.1f} < {MIN_INTERFACE_PLDDT}"

    epi_breakdown = "  |  ".join(
        f"{lbl}: {cnt}" for lbl, cnt in r["epitope_counts"].items()
    )

    rows.append({
        "pdb":              r["pdb"],
        "verdict":          verdict,
        "reason":           reason,
        "epitope_contacts": epi_contacts,
        "epitope_breakdown": epi_breakdown,
        "total_contacts":   r["total_contacts"],
        "interface_plddt":  iface_plddt,
        "binder_length":    r["binder_length"],
    })

df = pd.DataFrame(rows).sort_values(["verdict", "epitope_contacts"],
                                     ascending=[True, False]).reset_index(drop=True)

# ── Summary table ────────────────────────────────────────────────────────────
n_pass    = (df["verdict"] == "PASS").sum()
n_discard = (df["verdict"] == "DISCARD").sum()
n_error   = (df["verdict"] == "ERROR").sum()
n_total   = len(df)

print("=" * 50)
print("BINDER CONTACT FILTER — SUMMARY")
print("=" * 50)
print(f"  Total PDBs analyzed : {n_total}")
print(f"  PASS  (relevant)    : {n_pass}  ({100*n_pass/n_total:.1f}%)")
print(f"  DISCARD             : {n_discard}  ({100*n_discard/n_total:.1f}%)")
print(f"  ERROR               : {n_error}")
print(f"  Contact cutoff      : {CONTACT_CUTOFF} Å")
print(f"  Min interface pLDDT : {MIN_INTERFACE_PLDDT}")
print("=" * 50)
print()
print("Top 10 passing binders:")
pass_df = df[df["verdict"] == "PASS"].head(10)
print(pass_df[["pdb", "epitope_contacts", "epitope_breakdown",
               "total_contacts", "interface_plddt"]].to_string(index=False))

## 5. Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Summary pie ──────────────────────────────────────────────────────
ax = axes[0]
sizes  = [n_pass, n_discard] + ([n_error] if n_error else [])
labels = [f"PASS ({n_pass})", f"DISCARD ({n_discard})"] + \
         ([f"ERROR ({n_error})"] if n_error else [])
colors = ["#4CAF50", "#F44336", "#9E9E9E"]
ax.pie(sizes, labels=labels, colors=colors[:len(sizes)],
       autopct="%1.0f%%", startangle=90)
ax.set_title("Filter result")

# ── Plot 2: Epitope contacts distribution ────────────────────────────────────
ax = axes[1]
pass_vals    = df[df["verdict"] == "PASS"]["epitope_contacts"]
discard_vals = df[df["verdict"] == "DISCARD"]["epitope_contacts"]
bins = range(0, int(df["epitope_contacts"].max()) + 2)
ax.hist(discard_vals, bins=bins, color="#F44336", alpha=0.7, label="DISCARD")
ax.hist(pass_vals,    bins=bins, color="#4CAF50", alpha=0.7, label="PASS")
ax.axvline(MIN_CONTACTS - 0.5, color="black", linestyle="--",
           linewidth=1.2, label=f"Min contacts={MIN_CONTACTS}")
ax.set_xlabel("Epitope contacts")
ax.set_ylabel("Count")
ax.set_title("Epitope contact distribution")
ax.legend(fontsize=8)

# ── Plot 3: Interface pLDDT vs epitope contacts (passing only) ───────────────
ax = axes[2]
sc_pass = ax.scatter(
    pass_vals,
    df[df["verdict"] == "PASS"]["interface_plddt"],
    c=df[df["verdict"] == "PASS"]["total_contacts"],
    cmap="YlOrRd", s=50, edgecolors="none", alpha=0.85, label="PASS"
)
ax.scatter(
    discard_vals,
    df[df["verdict"] == "DISCARD"]["interface_plddt"],
    color="#aaaaaa", s=25, alpha=0.4, label="DISCARD"
)
plt.colorbar(sc_pass, ax=ax, label="Total contacts")
ax.axhline(MIN_INTERFACE_PLDDT, color="black", linestyle="--",
           linewidth=1, label=f"pLDDT≥{MIN_INTERFACE_PLDDT}")
ax.set_xlabel("Epitope contacts")
ax.set_ylabel("Interface pLDDT")
ax.set_title("Passing binders — quality vs. epitope contacts\n(color = total contacts)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("binder_filter_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved binder_filter_summary.png")

## 6. Save outputs + download

In [ ]:
import shutil, zipfile
from google.colab import files

# Full results spreadsheet (all PDBs)
all_csv = "binder_filter_all.csv"
df.to_csv(all_csv, index=False)
print(f"Full results   : {all_csv}  ({len(df)} rows)")

# Passing binders only
pass_csv = "binder_filter_passing.csv"
df[df["verdict"] == "PASS"].to_csv(pass_csv, index=False)
print(f"Passing binders: {pass_csv}  ({n_pass} rows)")

# Copy passing PDB files to a folder
pass_dir = "binders_passing"
os.makedirs(pass_dir, exist_ok=True)
for _, row in df[df["verdict"] == "PASS"].iterrows():
    if "path" in row and pd.notna(row["path"]):
        shutil.copy2(row["path"], os.path.join(pass_dir, row["pdb"]))

# Zip everything
zip_name = "binder_filter_results.zip"
with zipfile.ZipFile(zip_name, "w") as zf:
    zf.write(all_csv)
    zf.write(pass_csv)
    zf.write("binder_filter_summary.png")
    for f_name in os.listdir(pass_dir):
        zf.write(os.path.join(pass_dir, f_name))

print(f"\nDownloading {zip_name}")
files.download(zip_name)

## Output files

| File | Contents |
|------|----------|
| `binder_filter_all.csv` | Every PDB with verdict, epitope contact count, interface pLDDT, reason for discard |
| `binder_filter_passing.csv` | Passing binders only — feed this into Notebook 03 |
| `binders_passing/*.pdb` | PDB files of passing binders |
| `binder_filter_summary.png` | Pie chart + contact distribution + scatter plot |

## Notes

**Adjusting thresholds**:
- Increase `MIN_CONTACTS` to be stricter (e.g. 3 = binder must touch at least 3 epitope residues)
- Lower `CONTACT_CUTOFF` to 6Å for heavy-atom contacts (tighter); raise to 10Å to be more permissive
- Lower `MIN_INTERFACE_PLDDT` if AF2 confidence is generally low for your target (IDPs often score < 70)

**Chain IDs**: AF2 multimer usually assigns A=first input sequence, B=second. Check your PDB header if unsure — open any file and look at the ATOM records for the chain column.

**Next step**: pass `binder_filter_passing.csv` + `binders_passing/*.pdb` into Notebook 03 for ESM2 embedding, UMAP clustering, and control binder comparison.